# 03. Обучение по единому протоколу
Тот же модуль, что CLI tsprep-study. $\beta$ выбирается по validation на первом seed и фиксируется для повторов. Полный запуск: 144 выбранные модели и 24 дополнительных кандидата $\beta$; до 30 эпох.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "tsprep").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the tsprep project")
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)


In [ ]:
from datetime import datetime, timezone
from tsprep.study import run_study
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 200
SEEDS = [42, 43, 44]
DATASETS = ["synthetic", "aapl", "btc"]
STUDY_DIR = ROOT / "results" / datetime.now(timezone.utc).strftime("study_%Y%m%dT%H%M%S%fZ")
print(STUDY_DIR, DEVICE)


In [ ]:
# Each experiment stores full config, fitted preprocessing, inputs, hashes and weights.
# To resume, set STUDY_DIR to an existing study with exactly the same protocol/source.
run_study(ROOT, STUDY_DIR, epochs=EPOCHS, seeds=SEEDS, datasets=DATASETS, device=DEVICE, data_seed=2026, deterministic=True)


In [ ]:
import pandas as pd
runs = pd.read_csv(STUDY_DIR / "comparison_runs.csv")
runs.groupby(["dataset", "model"])[["mse", "forecast_mae", "forecast_vs_mean", "forecast_vs_median", "forecast_vs_raw", "forecast_vs_pca", "mse_vs_pca"]].agg(["mean", "std"])


Среднее/SD по seed и folds - описательная сводка, не доверительный интервал по независимым окнам. Положительное forecast_vs_mean / mse_vs_pca означает проигрыш baseline. Старые артефакты не смешиваются с новым исследованием.


In [ ]:
ALLOW_DOWNSTREAM_SOURCE_MISMATCH = False  # New study uses the current evaluator.

## Downstream: отдельные задачи и baseline
Сводки ниже разделены по dataset и fold. Среднее/SD описывают initialization seeds, а не независимые экономические истории. Baseline добавлен один раз на fold. `count` — число определённых измерений; пустая метрика не означает нулевое качество.

In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print
from tsprep.evaluation.study_results import load_study_tables, task_summary
from tsprep.evaluation.visualize import plot_task_metric

def show_task(frame, metrics):
    if frame.empty:
        print("Нет данных для этой задачи. Отсутствующая метрика не заменяется нулём.")
    else:
        display(task_summary(frame, metrics))

In [ ]:
study_tables = load_study_tables(STUDY_DIR, allow_source_mismatch=ALLOW_DOWNSTREAM_SOURCE_MISMATCH)
display(study_tables["availability"])
display(study_tables["support"])

### Прогнозирование
Ridge обучается на train embeddings, alpha выбирается на validation. Target — одно будущее преобразованное наблюдение в t+H; это не цена и не сумма доходностей. Сравниваются mean, median, last observation, raw-window Ridge и PCA+Ridge. Отрицательная `delta_mae_vs_*` означает преимущество относительно соответствующего baseline.

In [ ]:
show_task(study_tables["forecast"], ["mae", "r2", "validation_mae", "delta_mae_vs_mean",
                                    "delta_mae_vs_median", "delta_mae_vs_raw", "delta_mae_vs_pca"])
plot_task_metric(study_tables["forecast"], "mae", "Forecast MAE")
plt.show()

### Классификация и разделимость режимов
Бинарная классификация выполняется только при соответствующих метках; три synthetic-режима оцениваются через silhouette. Сравнение классификации включает PCA и train-majority baseline. USREC — ретроспективная метка периода, а не публикация в реальном времени. При одном test-классе AUC и часть остальных показателей не определены. Silhouette относительно истинных меток и KMeans — разные диагностические задачи.

In [ ]:
show_task(study_tables["classification"], ["accuracy", "balanced_accuracy", "f1", "precision", "recall",
                                          "pr_auc", "roc_auc", "false_positive_rate", "n_positive", "n_negative"])
show_task(study_tables["clustering"], ["silhouette", "n_clusters"])

### Аномалии: окна и концы окон
Оконный score = MSE + λ·KL, endpoint score = MSE последней точки. Порог калибруется только на validation. Смешанная validation-квантиль не задаёт 5% FPR на нормальных точках. Baseline: PCA и величина стандартизованного входа (raw magnitude). Оконные и endpoint-метрики не смешиваются. Число эпизодов относится только к наблюдаемым endpoints при заданном stride.

In [ ]:
show_task(study_tables["anomaly"], ["precision", "recall", "f1", "pr_auc", "false_positive_rate", "threshold",
    "positive_rate_test", "predicted_positive_rate", "n_positive", "n_negative", "n_events",
    "detected_events", "event_recall", "false_positive_points"])

### Reconstruction, clean signal и латентное пространство
Clean-эталон доступен только на синтетике. Совместно рассматриваются MSE и MAE относительно clean, исходный наблюдаемый вход и PCA. Сохранённые clean-метрики усредняют все элементы перекрывающихся окон; endpoint-разбор приведён отдельно ниже. Effective rank, active units и KL — диагностика, а variance ratio зависит от масштаба и не измеряет информацию.

In [ ]:
show_task(study_tables["reconstruction"], ["mse", "mae", "r2"])
show_task(study_tables["clean"], ["mse", "mae", "r2", "delta_mse_vs_input", "delta_mae_vs_input"])
show_task(study_tables["latent"], ["effective_rank", "active_units", "kl", "variance_ratio"])

Доступность метрик зависит от данных. Отсутствие бинарной классификации на synthetic с тремя режимами и anomaly labels на реальных CSV показано явно. Для самостоятельного сравнения observed/clean training targets используйте ноутбук 05; стандартный study здесь по-прежнему обучает observed → observed.